In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()
#Modification de la colonne Lt pour la transformer en variable numérique pour le random Forest
df.loc[df['Lt'] == 'R', 'Lt'] = 0
df.loc[df['Lt'] == 'G', 'Lt'] = 1

In [3]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','P2','Lt','Lq_reel','P4','P5','V','E2','P6_reel','E1']]

#On remplace les valeurs manquantes de P1_reel et Lq_reel par la première valeur non nulle de chaque essai selon les conseils de l'encadrante
#Cela permet de conserver d'avantage de lignes avant le dropna()
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])


df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['E1'])
y1 = df_ut['E1']

# 3. GESTION DES NaN (Le pansement propre avant le modèle)
# Un Random Forest de base dans scikit-learn ne tolère AUCUN NaN.
# Ici, on choisit d'imputer (remplacer) les valeurs manquantes par la médiane de la colonne.
imputer = SimpleImputer(strategy='median')
# On utilise fit_transform mais on le remet IMMÉDIATEMENT dans un DataFrame pour garder les index et noms de colonnes
X_clean = pd.DataFrame(imputer.fit_transform(X1), columns=X1.columns, index=X1.index)


X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

/var/folders/4m/c2py11t179n0qt638km54h440000gn/T/ipykernel_3812/3255010232.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])
/var/folders/4m/c2py11t179n0qt638km54h440000gn/T/ipykernel_3812/3255010232.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])


In [4]:
#Recherche des meilleurs hyperparamètres pour le Random Forest sur E1

param_grid = {
    'n_estimators': [100, 200, 600, 800],      # On cherche le bon nombre d'arbres
    'max_depth': [5, 8, 12, None],             # On cherche la bonne profondeur (PAS DE 'None' !)
    'min_samples_leaf': [2, 3, 5]        # On cherche à lisser plus ou moins les feuilles
}

# Modèle de base pour la recherche
rf_base = RandomForestRegressor(random_state=42)

print("Recherche des meilleurs paramètres pour E1 en cours...")
# cv=5 : Validation croisée en 5 blocs. n_jobs=-1 : Utilise tout le CPU.
grid_moy = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_moy.fit(X_train, y_train)
rf_moy = grid_moy.best_estimator_
print(f"Meilleurs paramètres E1 : {grid_moy.best_params_}")

print("--- 4. Évaluation ---")
# Prédictions (avec les meilleurs modèles trouvés automatiquement)
pred_moy = rf_moy.predict(X_test)

# Métriques pour E1
print(">> PERFORMANCES SUR E1 (La valeur moyenne) :")
print(f"MAE  : {mean_absolute_error(y_test, pred_moy):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_moy)):.4f}")
print(f"R2   : {r2_score(y_test, pred_moy):.4f}\n")


Recherche des meilleurs paramètres pour E1 en cours...
Meilleurs paramètres E1 : {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 600}
--- 4. Évaluation ---
>> PERFORMANCES SUR E1 (La valeur moyenne) :
MAE  : 0.0295
RMSE : 0.0441
R2   : 0.9412



In [5]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= None,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Ajout dans results
#results["RF"] = y_pred

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

# Exemple de sauvegarde dans un dataframe de résultats
results = pd.DataFrame({'Vraie_Valeur': y_test, 'Prediction_RF': y_pred})
print("Aperçu des prédictions :")
print(results.head())
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")





R2: 0.943455
MSE: 0.001874

Erreur Absolue Moyenne (MAE) : 0.03
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.04 


Aperçu des prédictions :
      Vraie_Valeur  Prediction_RF
1204      0.347255       0.347473
274       0.594557       0.584646
1400      0.367433       0.351917
518       0.983107       0.903806
2089      0.354294       0.370446


--- 4. Importance des Paramètres ---
V         : 32.0%
P2        : 25.3%
Lt        : 12.0%
P1_reel   : 8.4%
E2        : 7.7%
P5        : 6.6%
Lq_reel   : 4.6%
P6_reel   : 2.5%
P4        : 0.9%
